In [3]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 18.7 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-groq]


In [10]:
import os, json
from groq import Groq
from openai import RateLimitError, APIConnectionError

# Load API key from Colab secrets or environment
api_key = os.environ.get("GROQ_API_KEY") or os.environ.get("GroqApi")
if not api_key:
    from google.colab import userdata
    api_key = userdata.get("GroqApi")



In [11]:
client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-20b"

SYSTEM = """You are the receptionist for Sri Sai Clinic. Your job is to answer basic clinic questions and book patient appointments.
Facts: Clinic Name: Sri Sai Clinic, Doctor: Dr. Ramesh (General Physician), consultation fee: 300, Clinic hours: Monday to Saturday, 9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM, closed: Sunday.
Rules: You must collect three details to book an appointment: patient's name, Date, Time. Repeat and confirm all three details (name, date, and time) with the caller before finalizing the booking. Never give medical advice. For any medical questions, state that Dr. Ramesh can discuss it during the appointment, or offer to connect them to the reception staff.
Format: reply politely in under 60 words."""

In [12]:
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 2
bookings = []  # Stores confirmed appointments

def book_appointment(patient_name: str, date: str, time: str) -> str:
    """Save a confirmed patient appointment."""
    booking = {"patient_name": patient_name, "date": date, "time": time}
    bookings.append(booking)
    print(f"   [log] Appointment booked -> {booking}")
    return f"Appointment confirmed for {patient_name} on {date} at {time}."

clinic_tools = [
    {
        "type": "function",
        "function": {
            "name": "book_appointment",
            "description": "Finalize and save an appointment ONLY AFTER the caller has confirmed their name, date, and time.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_name": {"type": "string", "description": "Full name of the patient"},
                    "date": {"type": "string", "description": "Appointment date (e.g., 'Tomorrow', 'Monday', '2026-11-28')"},
                    "time": {"type": "string", "description": "Appointment time within clinic hours (9 AM-1 PM or 5 PM-8 PM)"}
                },
                "required": ["patient_name", "date", "time"]
            }
        }
    }
]

In [ ]:
def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            tools=clinic_tools,
            tool_choice="auto",
            temperature=0.3
        )
        msg = response.choices[0].message

        # Handle appointment tool call if triggered
        if msg.tool_calls:
            history.append(msg)
            for tool_call in msg.tool_calls:
                args = json.loads(tool_call.function.arguments)
                result = book_appointment(**args)
                history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

            recent_with_tool = [history[0]] + history[1:][-MAX_TURNS:]
            final_response = client.chat.completions.create(
                model=MODEL,
                messages=recent_with_tool,
                temperature=0.3
            )
            reply = final_response.choices[0].message.content
        else:
            reply = msg.content

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."

    history.append({"role": "assistant", "content": reply})
    return reply

# Start the receptionist chat loop
while True:
    text = input("Patient: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Receptionist: Thank you for contacting Sri Sai Clinic. Take care!")
        break
    print("Receptionist:", chat(text))

Receptionist: Sure! Just to confirm: your name is Akshay, you’d like an appointment tomorrow at 10 AM. Is that correct?
Receptionist: Dr. Ramesh is available Monday‑Saturday during two shifts: 9:00 AM‑1:00 PM and 5:00 PM‑8:00 PM. The clinic is closed on Sundays. Let me know a convenient date and time within these windows, and I’ll book your appointment.
Receptionist: Sri Sai Clinic is closed on Sundays to give our staff a rest day and to ensure we’re fully refreshed for the week ahead. We’re happy to see you Monday‑Saturday during our two shifts. Let me know a date and time that works for you!
Receptionist: Sri Sai Clinic is located at 123 Health Way, Greenfield Town, ZIP 45678. It’s just off Main St., near the community center. Feel free to call us at (555) 123‑4567 if you need directions or to book an appointment.
Receptionist: I’m sorry, but I don’t have that information. If you’d like to book an appointment, please let me know your full name, the date, and the time you’d like to vi